In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/sample_submission.csv
/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/valid.csv
/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/train.csv
/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/test.csv


In [2]:
TRAIN_PATH = '/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/train.csv'
TEST_PATH  = '/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/test.csv'
SAMPLE_PATH = '/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/sample_submission.csv'
VALID_PATH = '/kaggle/input/competitions/omahti-dsai-recruitment-bike-rush/valid.csv'

_tr = pd.read_csv(TRAIN_PATH); _te = pd.read_csv(TEST_PATH)
N_TRAIN = len(_tr)

# gabungkan train + test agar cleaning identik; baris test punya target NaN
df_train = pd.concat([_tr, _te], ignore_index=True)
print(df_train.shape, N_TRAIN)

(6665, 15) 4728


In [3]:
pd.set_option('display.max_rows', 100)

In [4]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6665 entries, 0 to 6664
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Id                    6665 non-null   object 
 1   month                 6665 non-null   int64  
 2   day_of_week           6665 non-null   int64  
 3   hour                  6665 non-null   int64  
 4   temperature_c         6498 non-null   float64
 5   humidity_pct          6525 non-null   float64
 6   wind_speed_ms         6504 non-null   float64
 7   visibility_10m        6494 non-null   float64
 8   dew_point_c           6485 non-null   float64
 9   solar_radiation_mjm2  6500 non-null   float64
 10  rainfall_mm           6510 non-null   float64
 11  snowfall_cm           6508 non-null   float64
 12  season                6665 non-null   object 
 13  holiday               6665 non-null   object 
 14  rented_bike_count     4728 non-null   float64
dtypes: float64(9), int64(

# Data Cleaning

In [5]:
# Check missing Value Value
df_train.isnull().sum()

Id                         0
month                      0
day_of_week                0
hour                       0
temperature_c            167
humidity_pct             140
wind_speed_ms            161
visibility_10m           171
dew_point_c              180
solar_radiation_mjm2     165
rainfall_mm              155
snowfall_cm              157
season                     0
holiday                    0
rented_bike_count       1937
dtype: int64

### Column Holiday

In [6]:
df_train['holiday'].value_counts()

holiday
No Holiday    6305
Holiday        360
Name: count, dtype: int64

In [7]:
df_train['holiday'] = df_train['holiday'].replace('No Holiday','0').replace('Holiday','1')
df_train['holiday'].value_counts()

holiday
0    6305
1     360
Name: count, dtype: int64

### Column Seasons

In [8]:
df_train['season'].value_counts()

season
Summer    1752
Spring    1704
Winter    1656
Autumn    1553
Name: count, dtype: int64

In [9]:
df_train.head(20)

,Id,month,day_of_week,hour,temperature_c,humidity_pct,wind_speed_ms,visibility_10m,dew_point_c,solar_radiation_mjm2,rainfall_mm,snowfall_cm,season,holiday,rented_bike_count
0,bk_754f4e10a46336bbbcb9,2,3,14,4.89,49.60,4.11,1174.74,-4.86,1.08,0.00,0.0,Winter,0,331.0
1,bk_b173e94a44f9021b66a8,2,6,15,-2.40,29.84,2.69,NaN,-17.57,1.38,0.00,0.0,Winter,0,181.0
2,bk_c2929a145ff14a2b66eb,7,4,14,29.60,62.87,2.77,1923.20,21.52,1.77,0.00,0.0,Summer,0,999.0
3,bk_bfb1af02d9c8a82b7051,11,2,21,4.33,32.29,1.20,1760.17,-11.32,0.00,0.00,0.0,Autumn,0,846.0
4,bk_4111f3bffd8d14296138,5,0,19,22.99,56.29,1.91,562.78,13.70,0.32,0.00,0.0,Spring,0,2236.0
5,bk_d39fb4499a3a94d60c8c,12,1,2,-1.95,69.08,3.43,1976.90,-7.12,NaN,0.00,0.0,Winter,0,85.0
6,bk_a1dd5f0b6d564a169af9,1,6,5,-2.67,NaN,0.96,1060.21,-11.75,0.00,0.00,0.0,Winter,0,32.0
7,bk_f8b57b7efee8aa892bf7,6,4,14,27.93,57.82,2.39,775.74,18.51,2.66,0.00,0.0,Summer,0,1175.0
8,bk_6727dc6ad5e07cc6bed1,10,6,14,18.88,31.48,2.34,1937.88,1.29,2.38,0.00,0.0,Autumn,0,1767.0
9,bk_85618834e6bc184299ef,1,2,22,-5.46,53.99,1.43,1949.48,-13.58,0.00,0.00,0.0,Winter,0,291.0


In [10]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6665 entries, 0 to 6664
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Id                    6665 non-null   object 
 1   month                 6665 non-null   int64  
 2   day_of_week           6665 non-null   int64  
 3   hour                  6665 non-null   int64  
 4   temperature_c         6498 non-null   float64
 5   humidity_pct          6525 non-null   float64
 6   wind_speed_ms         6504 non-null   float64
 7   visibility_10m        6494 non-null   float64
 8   dew_point_c           6485 non-null   float64
 9   solar_radiation_mjm2  6500 non-null   float64
 10  rainfall_mm           6510 non-null   float64
 11  snowfall_cm           6508 non-null   float64
 12  season                6665 non-null   object 
 13  holiday               6665 non-null   object 
 14  rented_bike_count     4728 non-null   float64
dtypes: float64(9), int64(